# 7.6 外部案例诊断报告

本节处理 xiaohetang、管弦乐等非标准 benchmark 素材。MUSDB18-HQ 可以计算 SI-SDR；这些外部案例通常没有标准 reference，因此重点转为能量分布、确认缺失 stem 的 false positive、uncertain stem 的人工复核线索，以及 stem taxonomy 与真实工程分轨之间的错配。


## 兼容性提示

- 请在前面章节已经建立的虚拟环境中运行本章 notebook（推荐 Python 3.11）。部分依赖包在 Python 3.13 上可能出现 `collections.Hashable` 等兼容性问题。
- 首次运行预训练模型时，工具会自动下载 checkpoint，需要一定时间，请耐心等待；本章不给出具体下载大小或耗时预估，因为不同网络与硬件差异较大。
- 若某模型依赖缺失，可将 `ENABLE_OPTIONAL_MODELS` 或对应运行开关设为 `0`，跳过该模型继续学习其余内容。


## 1. 路径与依赖

本单元设置输出目录，并导入外部案例诊断所需的音频读取、manifest 解析和绘图函数。


In [ ]:
import csv
import os
import sys
import tempfile
from pathlib import Path

# matplotlib/numba 缓存目录：用跨平台的系统临时目录（Windows 没有 /tmp）
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "mplconfig"))
os.environ.setdefault("NUMBA_CACHE_DIR", str(Path(tempfile.gettempdir()) / "numba_cache"))

# 路径推断：从 cwd 向上找含 CODE/chapter07/_common 的目录；NOTEBOOK_DIR 指向 CODE/chapter07/
_p = Path.cwd()
while not (_p / "CODE" / "chapter07" / "_common").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/chapter07/_common 的目录），请在项目内运行本 Notebook")
    _p = _parent
NOTEBOOK_DIR = _p / "CODE" / "chapter07"
CODE_ROOT = NOTEBOOK_DIR.parent
REPO_ROOT = CODE_ROOT.parent
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

FIG_DIR = NOTEBOOK_DIR / "output_figures"
FIG_DIR.mkdir(exist_ok=True)
TABLE_DIR = NOTEBOOK_DIR / "outputs" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
OUT_AUDIO_DIR = NOTEBOOK_DIR / "output_audio" / "07_4"

print("NOTEBOOK_DIR:", NOTEBOOK_DIR.relative_to(REPO_ROOT))
print("07_4 output_audio:", OUT_AUDIO_DIR.relative_to(REPO_ROOT))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from chapter07._common.audio_io import load_audio
from chapter07._common.external_diagnostics import (
    collect_manifest_output_paths,
    diagnostic_rows,
    discover_case_stem_paths,
    discover_output_audio_paths,
    find_case_mixture_path,
    output_mode_from_stems,
    parse_semicolon_list,
    read_external_manifest,
    repo_relative_path,
    resolve_case_dir,
    summarize_diagnostic_rows,
    write_diagnostic_rows,
    write_diagnostic_summary,
)
from chapter07._common.plotting import BAR_GRAY, FIGURE_SAVE_DPI, display_label, setup_plot_style
from chapter07._common.synthesis import make_synthetic_mixture


## 2. 小工具

这些函数负责表格写入、短标签显示、工程分轨读取和 `mixture - vocals` 派生伴奏。


In [ ]:
def truthy(value):
    return str(value or "").strip().lower() in {"1", "true", "yes", "y"}


def as_float(value, default=0.0):
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


def align_to_mixture(audio, mixture):
    n = min(len(audio), len(mixture))
    return audio[:n], mixture[:n]


def compact_label(value, limit=82):
    text = str(value)
    return text if len(text) <= limit else text[: limit - 1] + "…"


def write_rows(path, rows, fieldnames=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if fieldnames is None:
        fieldnames = []
        for row in rows:
            for key in row:
                if key not in fieldnames:
                    fieldnames.append(key)
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def print_rows(rows, columns, limit=12):
    if not rows:
        print("(no rows)")
        return
    display_rows = rows[:limit]
    widths = {
        col: min(max(len(str(row.get(col, ""))) for row in display_rows + [{col: col}]), 36)
        for col in columns
    }
    print(" | ".join(col.ljust(widths[col]) for col in columns))
    print("-+-".join("-" * widths[col] for col in columns))
    for row in display_rows:
        print(" | ".join(compact_label(row.get(col, ""), widths[col]).ljust(widths[col]) for col in columns))
    if len(rows) > limit:
        print(f"... {len(rows) - limit} more rows")


def load_owned_stems(case_dir, manifest_item, mixture_path, start, duration):
    stems = {}
    stem_paths = discover_case_stem_paths(
        case_dir,
        manifest_item,
        repo_root=REPO_ROOT,
        mixture_path=mixture_path,
    )
    for stem, path in stem_paths.items():
        stems[stem], _ = load_audio(path, sr=22050, mono=True, start=start, duration=duration)
    return stems


def derived_two_stem_view(mixture, stems):
    if "vocals" not in stems:
        return {}
    vocals, mix = align_to_mixture(stems["vocals"], mixture)
    return {
        "vocals": vocals,
        "accompaniment": (mix - vocals).astype(np.float32),
    }


def load_model_output_stems(stem_paths, duration):
    stems = {}
    for stem, path in sorted(stem_paths.items()):
        stems[stem], _ = load_audio(path, sr=22050, mono=True, duration=duration)
    return stems


## 3. 外部案例 manifest

`external_cases.csv` 描述外部素材路径、已有工程分轨、确认存在的 stem、确认缺失的 stem，以及需要人工复核的 uncertain stem。


In [ ]:
manifest_path = NOTEBOOK_DIR / "data_manifests" / "external_cases.csv"
if not manifest_path.exists():
    manifest_path = NOTEBOOK_DIR / "data_manifests" / "external_cases.example.csv"

manifest_rows = read_external_manifest(manifest_path)
case_ids = [row.get("case_id", "").strip() for row in manifest_rows if row.get("case_id", "").strip()]

print("manifest:", repo_relative_path(manifest_path, REPO_ROOT))
print("manifest rows:", len(manifest_rows))
print_rows(
    manifest_rows,
    ["case_id", "category", "reference_type", "expected_present", "expected_absent", "expected_uncertain"],
    limit=8,
)


## 4. 读取 07_4 模型输出

本单元读取 `07_4_output_audio_manifest.csv`。如果该 CSV 不存在，则扫描 `output_audio/07_4/<case_id>/<model_id>/`。后续诊断只分析已有音频，不下载 checkpoint，也不重新运行分离模型。


In [ ]:
output_manifest_path = TABLE_DIR / "07_4_output_audio_manifest.csv"
model_output_paths = collect_manifest_output_paths(output_manifest_path, REPO_ROOT, case_ids=case_ids)
output_source = "07_4_output_audio_manifest.csv"

if not model_output_paths:
    model_output_paths = discover_output_audio_paths(OUT_AUDIO_DIR, case_ids=case_ids)
    output_source = "output_audio/07_4 scan"

inventory_rows = []
for case_id, models in sorted(model_output_paths.items()):
    for model_id, stem_paths in sorted(models.items()):
        stems = sorted(stem_paths)
        inventory_rows.append(
            {
                "case_id": case_id,
                "model_id": model_id,
                "mode": output_mode_from_stems(stems),
                "stem_count": len(stems),
                "stems": ";".join(stems),
                "source": output_source,
            }
        )

inventory_path = TABLE_DIR / "07_6_model_output_inventory.csv"
write_rows(inventory_path, inventory_rows)
print("model output source:", output_source)
print("models found:", len(inventory_rows))
print("wrote:", repo_relative_path(inventory_path, REPO_ROOT))
print_rows(inventory_rows, ["case_id", "model_id", "mode", "stem_count", "stems"], limit=18)


## 5. stem 级诊断

本单元同时纳入三类对象：工程导出的 owned stems、由人声派生的 2-stem 视图、以及 `07_4` 已生成的模型输出。诊断指标包括 stem 能量比、输出求和重建误差、确认缺失 stem 能量和已知存在 stem 能量。


In [ ]:
rows = []
loaded_cases = []

for item in manifest_rows:
    case_id = item.get("case_id", "").strip()
    case_dir = resolve_case_dir(item.get("case_dir", ""), REPO_ROOT)
    mixture_path = find_case_mixture_path(case_dir, item, repo_root=REPO_ROOT)
    start = as_float(item.get("start_sec"), 0.0)
    duration = as_float(item.get("duration_sec"), 20.0)
    expected_present = parse_semicolon_list(item.get("expected_present"))
    expected_absent = parse_semicolon_list(item.get("expected_absent"))
    expected_uncertain = parse_semicolon_list(item.get("expected_uncertain"))
    notes = item.get("notes", "")

    if not case_id or mixture_path is None or not mixture_path.exists():
        print("skipping missing case:", case_id, repo_relative_path(case_dir, REPO_ROOT))
        continue

    mixture, _ = load_audio(mixture_path, sr=22050, mono=True, start=start, duration=duration)
    loaded_cases.append(case_id)
    print("case:", case_id, "input:", repo_relative_path(mixture_path, REPO_ROOT))

    owned_stems = load_owned_stems(case_dir, item, mixture_path, start, duration)
    if owned_stems:
        rows.extend(
            diagnostic_rows(
                case_id,
                "owned_reference",
                "owned_stems",
                mixture,
                owned_stems,
                reference_type=item.get("reference_type", "owned_project_stems"),
                expected_present=expected_present,
                expected_absent=expected_absent,
                expected_uncertain=expected_uncertain,
                notes=notes,
            )
        )

    if truthy(item.get("derive_accompaniment")):
        two_stem = derived_two_stem_view(mixture, owned_stems)
        if two_stem:
            rows.extend(
                diagnostic_rows(
                    case_id,
                    "owned_reference",
                    "owned_2stem_derived",
                    mixture,
                    two_stem,
                    reference_type="owned_2stem_derived",
                    expected_present=["vocals", "accompaniment"],
                    expected_absent=expected_absent,
                    expected_uncertain=expected_uncertain,
                    notes=notes + " Derived accompaniment = mixture - vocals.",
                )
            )

    for model_id, stem_paths in sorted(model_output_paths.get(case_id, {}).items()):
        estimates = load_model_output_stems(stem_paths, duration)
        if not estimates:
            continue
        rows.extend(
            diagnostic_rows(
                case_id,
                model_id,
                output_mode_from_stems(estimates.keys()),
                mixture,
                estimates,
                reference_type="model_output_no_reference",
                expected_present=expected_present,
                expected_absent=expected_absent,
                expected_uncertain=expected_uncertain,
                notes=notes,
            )
        )

print("loaded cases:", loaded_cases)
print("diagnostic rows from real cases:", len(rows))


## 6. 无外部音频时的 synthetic fallback

当外部素材和 `07_4` 输出都不可用时，本单元生成一个小型 no-vocal 案例，保证诊断表和图形仍可执行。


In [ ]:
if not rows:
    demo = make_synthetic_mixture(sr=22050, duration=6.0, seed=66)
    mixture = demo["mixture"]
    estimates = {
        "vocals": 0.04 * demo["harmonic"],
        "drums": demo["percussive"],
        "bass": demo["bass"],
        "other": demo["harmonic"],
    }
    rows.extend(
        diagnostic_rows(
            "synthetic_orchestra_no_vocal_demo",
            "synthetic_probe",
            "4stems",
            mixture,
            estimates,
            reference_type="synthetic_no_reference_demo",
            expected_present=["drums", "bass", "other"],
            expected_absent=["vocals"],
            expected_uncertain=[],
            notes="Fallback demo because external audio or 07_4 outputs are not present.",
        )
    )
    print("created synthetic external diagnostics demo")

print("total diagnostic rows:", len(rows))


## 7. 保存 stem 明细与 case/model 汇总

stem 明细表保留每个输出 stem 的诊断值；case/model 汇总表把重建误差、确认缺失 stem 能量和 uncertain stem 能量聚合成复核优先级。


In [ ]:
diagnostic_path = TABLE_DIR / "07_6_external_diagnostics.csv"
write_diagnostic_rows(diagnostic_path, rows)

summary_rows = summarize_diagnostic_rows(rows)
summary_path = TABLE_DIR / "07_6_case_model_summary.csv"
write_diagnostic_summary(summary_path, summary_rows)

print("wrote:", repo_relative_path(diagnostic_path, REPO_ROOT))
print("wrote:", repo_relative_path(summary_path, REPO_ROOT))
print_rows(
    summary_rows,
    [
        "case_id",
        "model",
        "mode",
        "stem_count",
        "reconstruction_error",
        "absent_energy_ratio_max",
        "absent_stem_with_max",
        "uncertain_energy_ratio_sum",
        "review_priority",
    ],
    limit=20,
)


## 8. 诊断图

三张图分别显示最高能量 stem、确认缺失 stem 的相对混音能量，以及按 case/model 聚合后的复核风险。


In [ ]:
setup_plot_style()

ENERGY_TOP_N = 28
ABSENT_TOP_N = 28
SUMMARY_TOP_N = 24
LABEL_FONT_SIZE = 8

plot_rows = sorted(rows, key=lambda row: as_float(row.get("energy_ratio")), reverse=True)[:ENERGY_TOP_N]
labels = [
    compact_label(f"{row['case_id']} | {row['model']} | {display_label(row['stem'])}", 92)
    for row in plot_rows
]
values = [as_float(row.get("energy_ratio")) for row in plot_rows]
y = np.arange(len(values))

fig, ax = plt.subplots(figsize=(10.5, max(4.2, 0.36 * max(len(labels), 1))))
if values:
    ax.barh(y, values, color=BAR_GRAY)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=LABEL_FONT_SIZE)
    ax.invert_yaxis()
    ax.set_xlabel("相对于混合信号的能量比")
else:
    ax.text(0.5, 0.5, "暂无可绘制的 stem 能量数据", ha="center", va="center")
    ax.set_axis_off()
ax.set_title("外部案例 stem 能量比 Top-N")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(FIG_DIR / "07_6_external_energy_ratios.png", bbox_inches="tight", dpi=FIGURE_SAVE_DPI)
plt.show()
plt.close(fig)

absent_rows = sorted(
    [row for row in rows if row.get("absent_energy_ratio") != ""],
    key=lambda row: as_float(row.get("absent_energy_ratio")),
    reverse=True,
)[:ABSENT_TOP_N]
labels = [
    compact_label(f"{row['case_id']} | {row['model']} | {display_label(row['stem'])}", 92)
    for row in absent_rows
]
values = [as_float(row.get("absent_energy_ratio")) for row in absent_rows]
y = np.arange(len(values))

fig, ax = plt.subplots(figsize=(10.5, max(3.8, 0.38 * max(len(labels), 1))))
if values:
    ax.barh(y, values, color="0.28")
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=LABEL_FONT_SIZE)
    ax.invert_yaxis()
    ax.set_xlabel("确认缺失 stem 的能量比")
else:
    ax.text(0.5, 0.5, "manifest 未标注确认缺失的 stem", ha="center", va="center")
    ax.set_axis_off()
ax.set_title("确认缺失 stem 的相对混音能量")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(FIG_DIR / "07_6_absent_stem_energy.png", bbox_inches="tight", dpi=FIGURE_SAVE_DPI)
plt.show()
plt.close(fig)

priority_rank = {"high": 0, "medium": 1, "low": 2}
summary_plot_rows = sorted(
    summary_rows,
    key=lambda row: (
        priority_rank.get(str(row.get("review_priority")), 3),
        -as_float(row.get("absent_energy_ratio_sum")),
        -as_float(row.get("uncertain_energy_ratio_sum")),
        -as_float(row.get("reconstruction_error")),
    ),
)[:SUMMARY_TOP_N]
labels = [
    compact_label(f"{row['case_id']} | {row['model']} | {row['mode']}", 92)
    for row in summary_plot_rows
]
absent_values = [as_float(row.get("absent_energy_ratio_sum")) for row in summary_plot_rows]
uncertain_values = [as_float(row.get("uncertain_energy_ratio_sum")) for row in summary_plot_rows]
recon_values = [as_float(row.get("reconstruction_error")) for row in summary_plot_rows]
y = np.arange(len(labels))
bar_height = 0.24

fig, ax = plt.subplots(figsize=(10.8, max(4.2, 0.42 * max(len(labels), 1))))
if labels:
    ax.barh(y - bar_height, absent_values, height=bar_height, color="0.20", label="确认缺失 stem 能量")
    ax.barh(y, uncertain_values, height=bar_height, color="0.50", label="uncertain stem 能量")
    ax.barh(y + bar_height, recon_values, height=bar_height, color="0.75", label="重建误差")
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=LABEL_FONT_SIZE)
    ax.invert_yaxis()
    ax.set_xlabel("诊断值")
    ax.legend(loc="lower right", frameon=False)
else:
    ax.text(0.5, 0.5, "暂无 case/model 汇总数据", ha="center", va="center")
    ax.set_axis_off()
ax.set_title("外部案例 case/model 复核优先级")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(FIG_DIR / "07_6_model_diagnostic_summary.png", bbox_inches="tight", dpi=FIGURE_SAVE_DPI)
plt.show()
plt.close(fig)


## 9. 结论读取方式

- `07_6_external_diagnostics.csv` 是 stem 级明细，适合追踪某个模型把能量分到哪个 stem。
- `07_6_case_model_summary.csv` 是 case/model 级摘要，适合选择优先听测的模型与片段。
- `absent_energy_ratio` 只解释 manifest 中确认缺失的 stem；`uncertain_energy_ratio_sum` 只提示需要人工复核，不等同于错误。
- xiaohetang 的钢琴 stem 可作为工程素材参照，但带混响、软音源音色和遮蔽时，不应与 MUSDB18-HQ 的标准钢琴/其他 stem 指标混排。
- 管弦乐案例的重点是 taxonomy mismatch：流行音乐模型可能把不同工程乐器组压缩到 `other`，或把某些成分分到 `vocals/drums/bass`。判断 false positive 时必须使用当前分析区间的参考活动状态。本案例整曲虽含 choirs，但 0~30 秒参考近似静默。
